# GNN-BERT: Music Context Understanding Demo

**Course**: CSE425 / EEE474 / CSE715 Neural Networks Project  
**System**: Hybrid Graph Neural Network + BERT for Music Context Understanding  

This notebook demonstrates end-to-end inference as required by Section 10 of the project specification:
1. **Audio Feature Extraction**: Log-Mel Spectrogram (128 bins), Chroma (12 bins), and MFCCs (20 bins) per segment.
2. **Relational Graph Construction**: Segment graphs G=(V, E) with temporal adjacency and feature similarity edges.
3. **Contextual Text Encoding**: Context tokenization and representations with DistilBERT.
4. **Cross-Attention Fusion (Task 3)**: Predicting multi-label context tags and valence/arousal emotion scores.
5. **Contrastive Retrieval (Task 4)**: Bidirectional similarity scoring between audio structure and captions.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
from transformers import AutoTokenizer

from src.audio_features import AudioFeatureExtractor, generate_synthetic_audio, set_seed, load_config
from src.graph_builder import build_segment_graph
from src.fusion_model import MusicGNNBertFusionModel
from src.contrastive import MusicContrastiveDualEncoder

config = load_config('config.yaml')
seed = config.get('seed', 42)
set_seed(seed)
print('Environment initialized. Using device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. Audio Feature Extraction & Segmentation
Extract log-mel spectrogram (128 bins) and chroma (12 bins) at 22,050 Hz and segment into 8s windows.

In [ ]:
extractor = AudioFeatureExtractor.from_config('config.yaml')
audio = generate_synthetic_audio(duration=30.0, sample_rate=extractor.sample_rate, seed=seed)
duration_sec = len(audio) / extractor.sample_rate
print(f'Audio sample rate: {extractor.sample_rate} Hz, Duration: {duration_sec:.1f} s')

mel = extractor.compute_log_mel_spectrogram(audio)
chroma = extractor.compute_chroma(audio)
node_features = extractor.extract_segment_features(audio)
print(f'Extracted {node_features.shape[0]} segments, each with {node_features.shape[1]}-dim feature vector.')

## 2. Music Structure Graph Construction
Construct G = (V, E) connecting segments with temporal edges (i <-> i+1) and cosine similarity edges.

In [ ]:
graph = build_segment_graph(
    segment_features=node_features,
    similarity_threshold=config.get('graph', {}).get('similarity_threshold', 0.7),
    add_temporal_edges=config.get('graph', {}).get('add_temporal_edges', True)
)
print(f'Constructed PyG Graph: {graph.num_nodes} nodes, {graph.num_edges} edges, feature dim: {graph.x.shape[1]}')

## 3. End-to-End GNN-BERT Fusion Inference (Task 3)
Pass the audio structure graph and a descriptive text query into the cross-attention fusion model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
sample_caption = 'slow melancholic piano ballad with soft classical strings and emotional melodic mood'
encoded_text = tokenizer(sample_caption, max_length=128, padding='max_length', truncation=True, return_tensors='pt')

fusion_model = MusicGNNBertFusionModel.from_config('config.yaml', in_channels=node_features.shape[1], num_tags=50)
fusion_model.eval()
with torch.no_grad():
    out = fusion_model(graph_data=graph, input_ids=encoded_text['input_ids'], attention_mask=encoded_text['attention_mask'])

tag_probs = out['tag_probs'].squeeze(0).numpy()
valence, arousal = out['emotion_preds'].squeeze(0).numpy()
print(f'Predicted Valence: {valence:.2f} / 9.0, Predicted Arousal: {arousal:.2f} / 9.0')
top_5 = np.argsort(tag_probs)[::-1][:5]
print('Top-5 Predicted Tag Indices:', [int(idx) for idx in top_5])
print('Top-5 Tag Probabilities:', [round(float(tag_probs[idx]), 4) for idx in top_5])

## 4. Cross-Modal Retrieval Demo (Task 4)
Compute cosine similarity between the audio graph and candidate text captions.

In [ ]:
contrastive_model = MusicContrastiveDualEncoder.from_config('config.yaml', in_channels=node_features.shape[1], proj_dim=128)
contrastive_model.eval()
candidate_captions = [
    'slow melancholic piano ballad with soft classical strings',
    'energetic electronic dance beat with heavy distorted bass',
    'loud distorted rock guitars with aggressive fast drum rhythm',
    'calm ambient acoustic folk music with gentle soothing melody'
]
encoded_candidates = tokenizer(candidate_captions, max_length=64, padding='max_length', truncation=True, return_tensors='pt')
with torch.no_grad():
    audio_emb = contrastive_model.encode_audio(graph)
    text_emb = contrastive_model.encode_text(encoded_candidates['input_ids'], encoded_candidates['attention_mask'])
    similarities = torch.matmul(audio_emb, text_emb.T).squeeze(0).numpy()

ranking = np.argsort(similarities)[::-1]
for rank, idx in enumerate(ranking, start=1):
    print(f'Rank {rank}: [Sim: {similarities[idx]:.4f}] {candidate_captions[idx]}')